# Solution B3 · Multiclass classification: handwritten digits

**Type**: Implement with scikit-learn

## Objective

Solve a 10-class problem with a logistic: choose `C` with `LogisticRegressionCV`, compare the multinomial approach with **one-vs-rest**, summarize the
performance with the right averages (macro, per class) and identify the main confusions.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import confusion_matrix, f1_score, log_loss, recall_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(digits.data, digits.target, test_size=0.3, stratify=digits.target, random_state=0)

## Commented solution

In [ ]:
cv_model = make_pipeline(StandardScaler(), LogisticRegressionCV(Cs=np.logspace(-4, 1, 6), cv=5, scoring="neg_log_loss", max_iter=3000)).fit(X_tr, y_tr)
chosen_C = float(cv_model[-1].C_[0])
print("C chosen in cross-validation:", chosen_C, "| values tried:", cv_model[-1].Cs_)

In [ ]:
multinomial = make_pipeline(StandardScaler(), LogisticRegression(C=chosen_C, max_iter=3000)).fit(X_tr, y_tr)
ovr = make_pipeline(StandardScaler(), OneVsRestClassifier(LogisticRegression(C=chosen_C, max_iter=3000))).fit(X_tr, y_tr)
table = pd.DataFrame(index=["multinomial", "ovr"], columns=["accuracy", "macro_f1", "log_loss"], dtype=float)
for name, m in {"multinomial": multinomial, "ovr": ovr}.items():
    table.loc[name] = [m.score(X_te, y_te), f1_score(y_te, m.predict(X_te), average="macro"), log_loss(y_te, m.predict_proba(X_te))]
table.round(4)

In [ ]:
pred = multinomial.predict(X_te)
per_class_recall = recall_score(y_te, pred, average=None)
hardest_class = int(np.argmin(per_class_recall))
cm = confusion_matrix(y_te, pred)
errors = cm.copy(); np.fill_diagonal(errors, 0)
main_confusion = tuple(int(i) for i in np.unravel_index(errors.argmax(), errors.shape))
coef_shape = multinomial[-1].coef_.shape
prob_sum = float(multinomial.predict_proba(X_te[:1]).sum())
print("recall per class:", per_class_recall.round(3), "| the hardest:", hardest_class)
print("most frequent error (true, predicted):", main_confusion, "with", errors.max(), "cases")

## Check

In [ ]:
with verify():
    check(isinstance(chosen_C, float) and any(np.isclose(chosen_C, c) for c in np.logspace(-4, 1, 6)), "C chosen among the grid values", "chosen_C = float(cv_model[-1].C_[0])")
    check(table.notna().all().all() and table.loc["multinomial", "accuracy"] > 0.95, "multinomial accuracy above 0.95", "use the same C for both models")
    check(table.loc["multinomial", "log_loss"] < table.loc["ovr", "log_loss"], "the multinomial has a better log-loss than one-vs-rest", "the multinomial probabilities sum to 1 by construction, the ovr ones are normalized afterwards")
    check(close(table.loc["multinomial", "macro_f1"], f1_score(y_te, multinomial.predict(X_te), average="macro")), "correct macro F1", "average='macro'")
    check(len(per_class_recall) == 10 and hardest_class == int(np.argmin(per_class_recall)) and hardest_class == 8, "the hardest digit is the 8", "recall_score(..., average=None)")
    check(main_confusion == (8, 1), "the most frequent error is 8 mistaken for 1", "confusion matrix with the diagonal zeroed, then argmax")
    check(tuple(coef_shape) == (10, 64) and close(prob_sum, 1.0), "coef_ has one row per class and the probabilities sum to 1", "shape (10, 64)")
done()

## What you learned

- In the multiclass case `coef_` has one row per class and `predict_proba` is a softmax: the probabilities sum to 1 **by construction**. `OneVsRestClassifier` trains 10 independent
  binary models and normalizes afterwards: its probabilities are less coherent (here the log-loss is about 35% worse, 0.14 against 0.10).
- `LogisticRegressionCV` chooses `C` on a grid with cross-validation and refits the model on the whole training set: it is the equivalent of `GridSearchCV` for a logistic, much faster
  because along the grid it reuses the previous solutions (warm start).
- **The accuracy (0.97) hides the differences between classes**: the recall goes from 0.92 (the 8) to 1.0 (0 and 7). Looking at `average=None` and the confusion matrix tells you where to intervene;
  the macro F1 treats all the classes alike, which is what you want when the rare classes matter.
- With only 540 test examples, one error more or less moves the accuracy by 0.2 points: differences between models of this order of magnitude are noise.

## Common pitfall

Reporting only the accuracy on a multiclass problem: a class with recall 0.6 can vanish in the average if it is rare. For every multiclass problem always look at
recall and precision **per class** before trusting a single number.

## Interview question

> *Click on a question to show its answer.*

<details><summary>What is the difference between macro, micro and weighted average?</summary>

Macro: simple mean of the per-class metrics, every class weighs the same (sensitive to rare classes). Micro: counts TP, FP, FN over all the classes together
(for single-label multiclass it coincides with the accuracy). Weighted: mean weighted by the number of examples of each class. I use macro when the rare
classes matter, micro or accuracy when the total is what matters.

</details>